In [1]:
print('hello world')

hello world


In [2]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from joblib import Parallel, delayed
import matplotlib.pyplot as plt 
import seaborn as sns
%matplotlib inline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
import scipy.stats as stats
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer, LabelEncoder,PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [4]:
df=pd.read_csv('agronimy mandi dataset.csv')
des_df=pd.read_csv('des dataset.csv')
apy=pd.read_csv('APY.csv')
apy_2024_demo=pd.read_csv('2024_demo.csv')
apy_2025_demo=pd.read_csv('2025_demo.csv')

In [5]:
print("Mandi states:")
print(sorted(df["State"].dropna().unique()))

print("\n2024 Demo states:")
print(sorted(apy_2024_demo["State"].dropna().unique()))

print("\n2025 Demo states:")
print(sorted(apy_2025_demo["State"].dropna().unique()))

Mandi states:
['Andhra Pradesh', 'Gujarat', 'Kerala', 'Madhya Pradesh', 'Punjab', 'Rajasthan', 'Uttar Pradesh', 'West Bengal']

2024 Demo states:
['Andaman and Nicobar Island', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'CHANDIGARH', 'Chhattisgarh', 'Dadra and Nagar Haveli', 'Daman and Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu and Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']

2025 Demo states:
['Andaman and Nicobar Island', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'CHANDIGARH', 'Chhattisgarh', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu and Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Laddak', 'Madhya Pradesh', 'Maharashtra', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', '

In [6]:
# States present in ALL THREE datasets

mandi_states = set(df["State"].dropna().str.strip())
demo24_states = set(apy_2024_demo["State"].dropna().str.strip())
demo25_states = set(apy_2025_demo["State"].dropna().str.strip())

common_states = sorted(
    mandi_states & demo24_states & demo25_states
)

print("States common to all 3 datasets:", len(common_states))
print(common_states)

States common to all 3 datasets: 8
['Andhra Pradesh', 'Gujarat', 'Kerala', 'Madhya Pradesh', 'Punjab', 'Rajasthan', 'Uttar Pradesh', 'West Bengal']


In [7]:
# Create normalized State and District columns

mandi_locations = (
    df[["State", "District Name"]]
    .drop_duplicates()
    .copy()
)

mandi_locations["State_clean"] = (
    mandi_locations["State"]
    .astype(str)
    .str.strip()
    .str.lower()
)

mandi_locations["District_clean"] = (
    mandi_locations["District Name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

demo24_locations = (
    apy_2024_demo[["State", "District "]]
    .drop_duplicates()
    .copy()
)

demo24_locations["State_clean"] = (
    demo24_locations["State"]
    .astype(str)
    .str.strip()
    .str.lower()
)

demo24_locations["District_clean"] = (
    demo24_locations["District "]
    .astype(str)
    .str.strip()
    .str.lower()
)

demo25_locations = (
    apy_2025_demo[["State", "District "]]
    .drop_duplicates()
    .copy()
)

demo25_locations["State_clean"] = (
    demo25_locations["State"]
    .astype(str)
    .str.strip()
    .str.lower()
)

demo25_locations["District_clean"] = (
    demo25_locations["District "]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Find locations present in all three datasets
mandi_set = set(
    zip(mandi_locations["State_clean"], mandi_locations["District_clean"])
)

demo24_set = set(
    zip(demo24_locations["State_clean"], demo24_locations["District_clean"])
)

demo25_set = set(
    zip(demo25_locations["State_clean"], demo25_locations["District_clean"])
)

common_locations = sorted(
    mandi_set & demo24_set & demo25_set
)

print("Mandi locations:", len(mandi_set))
print("2024 Demo locations:", len(demo24_set))
print("2025 Demo locations:", len(demo25_set))
print("Common locations in ALL THREE:", len(common_locations))

print("\nFirst 50 common locations:")
for state, district in common_locations[:50]:
    print(state, "—", district)

Mandi locations: 270
2024 Demo locations: 693
2025 Demo locations: 700
Common locations in ALL THREE: 197

First 50 common locations:
andhra pradesh — anantapur
andhra pradesh — east godavari
andhra pradesh — guntur
andhra pradesh — krishna
andhra pradesh — kurnool
andhra pradesh — west godavari
gujarat — amreli
gujarat — anand
gujarat — bharuch
gujarat — bhavnagar
gujarat — botad
gujarat — dang
gujarat — devbhumi dwarka
gujarat — gandhinagar
gujarat — gir somnath
gujarat — jamnagar
gujarat — kachchh
gujarat — kheda
gujarat — morbi
gujarat — narmada
gujarat — navsari
gujarat — patan
gujarat — porbandar
gujarat — rajkot
gujarat — surat
gujarat — surendranagar
gujarat — valsad
kerala — alappuzha
kerala — ernakulam
kerala — idukki
kerala — kannur
kerala — kollam
kerala — kottayam
kerala — malappuram
kerala — pathanamthitta
kerala — thiruvananthapuram
kerala — wayanad
madhya pradesh — agar malwa
madhya pradesh — alirajpur
madhya pradesh — ashoknagar
madhya pradesh — balaghat
madhya pradesh

In [8]:
# ---------- MANDI ----------

mandi_sdc = df[["State", "District Name", "Commodity"]].drop_duplicates().copy()

mandi_sdc["State_clean"] = (
    mandi_sdc["State"].astype(str).str.strip().str.lower()
)

mandi_sdc["District_clean"] = (
    mandi_sdc["District Name"].astype(str).str.strip().str.lower()
)

mandi_sdc["Crop_clean"] = (
    mandi_sdc["Commodity"].astype(str).str.strip().str.lower()
)


# ---------- APY 2024 DEMO ----------

demo24_sdc = apy_2024_demo[["State", "District ", "Crop"]].drop_duplicates().copy()

demo24_sdc["State_clean"] = (
    demo24_sdc["State"].astype(str).str.strip().str.lower()
)

demo24_sdc["District_clean"] = (
    demo24_sdc["District "].astype(str).str.strip().str.lower()
)

demo24_sdc["Crop_clean"] = (
    demo24_sdc["Crop"].astype(str).str.strip().str.lower()
)


# ---------- APY 2025 DEMO ----------

demo25_sdc = apy_2025_demo[["State", "District ", "Crop"]].drop_duplicates().copy()

demo25_sdc["State_clean"] = (
    demo25_sdc["State"].astype(str).str.strip().str.lower()
)

demo25_sdc["District_clean"] = (
    demo25_sdc["District "].astype(str).str.strip().str.lower()
)

demo25_sdc["Crop_clean"] = (
    demo25_sdc["Crop"].astype(str).str.strip().str.lower()
)


# ---------- NORMALIZE OBVIOUS CROP NAME DIFFERENCES ----------

crop_mapping = {
    "bajra(pearl millet/cumbu)": "bajra",
    "jowar(sorghum)": "jowar",
    "lentil (masur)(whole)": "lentil",
    "soyabean": "soybean",
    "green gram (moong)(whole)": "moong",
    "arhar (tur/red gram)(whole)": "tur",
    "mustard": "rapeseed & mustard"
}

mandi_sdc["Crop_clean"] = mandi_sdc["Crop_clean"].replace(crop_mapping)


# ---------- CREATE SETS ----------

mandi_set = set(zip(
    mandi_sdc["State_clean"],
    mandi_sdc["District_clean"],
    mandi_sdc["Crop_clean"]
))

demo24_set = set(zip(
    demo24_sdc["State_clean"],
    demo24_sdc["District_clean"],
    demo24_sdc["Crop_clean"]
))

demo25_set = set(zip(
    demo25_sdc["State_clean"],
    demo25_sdc["District_clean"],
    demo25_sdc["Crop_clean"]
))


# ---------- COMMON TO ALL THREE ----------

common_sdc = sorted(
    mandi_set & demo24_set & demo25_set
)

print("Mandi State-District-Crop combinations:", len(mandi_set))
print("2024 Demo combinations:", len(demo24_set))
print("2025 Demo combinations:", len(demo25_set))
print("Common combinations in ALL THREE:", len(common_sdc))

print("\nFirst 50:")
for state, district, crop in common_sdc[:50]:
    print(f"{state} | {district} | {crop}")

Mandi State-District-Crop combinations: 2970
2024 Demo combinations: 15360
2025 Demo combinations: 15822
Common combinations in ALL THREE: 532

First 50:
andhra pradesh | anantapur | groundnut
andhra pradesh | anantapur | maize
andhra pradesh | krishna | maize
andhra pradesh | kurnool | bajra
andhra pradesh | kurnool | groundnut
andhra pradesh | kurnool | jowar
andhra pradesh | kurnool | maize
andhra pradesh | west godavari | maize
gujarat | amreli | bajra
gujarat | amreli | garlic
gujarat | amreli | groundnut
gujarat | amreli | jowar
gujarat | amreli | maize
gujarat | anand | bajra
gujarat | anand | maize
gujarat | bharuch | bajra
gujarat | bharuch | jowar
gujarat | bharuch | maize
gujarat | bhavnagar | bajra
gujarat | bhavnagar | garlic
gujarat | bhavnagar | groundnut
gujarat | bhavnagar | maize
gujarat | botad | bajra
gujarat | botad | groundnut
gujarat | devbhumi dwarka | bajra
gujarat | devbhumi dwarka | groundnut
gujarat | gandhinagar | bajra
gujarat | gir somnath | bajra
gujarat

In [9]:
# Count unique mandi trading days for every State-District-Crop combination

mandi_coverage = (
    df.groupby(
        ["State", "District Name", "Commodity"]
    )["Price Date"]
    .nunique()
    .reset_index(name="unique_days")
)

# Create the same normalized keys
mandi_coverage["State_clean"] = (
    mandi_coverage["State"].astype(str).str.strip().str.lower()
)

mandi_coverage["District_clean"] = (
    mandi_coverage["District Name"].astype(str).str.strip().str.lower()
)

mandi_coverage["Crop_clean"] = (
    mandi_coverage["Commodity"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(crop_mapping)
)

# Keep only combinations common to all 3 datasets
common_df = pd.DataFrame(
    common_sdc,
    columns=["State_clean", "District_clean", "Crop_clean"]
)

ranked_candidates = common_df.merge(
    mandi_coverage[
        [
            "State_clean",
            "District_clean",
            "Crop_clean",
            "State",
            "District Name",
            "Commodity",
            "unique_days"
        ]
    ],
    on=["State_clean", "District_clean", "Crop_clean"],
    how="left"
)

# Rank by number of available mandi days
ranked_candidates = ranked_candidates.sort_values(
    "unique_days",
    ascending=False
).reset_index(drop=True)

print("Total common candidates:", len(ranked_candidates))

print("\nTop 30 candidates:")
display(
    ranked_candidates[
        ["State", "District Name", "Commodity", "unique_days"]
    ].head(30)
)

Total common candidates: 532

Top 30 candidates:


,State,District Name,Commodity,unique_days
0,West Bengal,Bankura,Wheat,365
1,Uttar Pradesh,Mainpuri,Wheat,365
2,West Bengal,Birbhum,Wheat,364
3,Uttar Pradesh,Fatehpur,Wheat,364
4,Uttar Pradesh,Gonda,Banana,362
5,Uttar Pradesh,Hardoi,Wheat,362
6,Uttar Pradesh,Shahjahanpur,Wheat,362
7,West Bengal,Paschim Bardhaman,Wheat,362
8,Uttar Pradesh,Balrampur,Wheat,360
9,Uttar Pradesh,Shamli,Garlic,358


In [10]:
# Take top 25 mandi candidates
top_candidates = ranked_candidates.head(25).copy()

# Function to count APY records for a State-District-Crop
def get_apy_count(apy_data, state, district, crop):
    temp = apy_data.copy()

    temp["State_clean"] = (
        temp["State"].astype(str).str.strip().str.lower()
    )
    temp["District_clean"] = (
        temp["District "].astype(str).str.strip().str.lower()
    )
    temp["Crop_clean"] = (
        temp["Crop"].astype(str).str.strip().str.lower()
    )

    return len(
        temp[
            (temp["State_clean"] == state) &
            (temp["District_clean"] == district) &
            (temp["Crop_clean"] == crop)
        ]
    )

# Calculate APY record counts
top_candidates["demo24_records"] = top_candidates.apply(
    lambda row: get_apy_count(
        apy_2024_demo,
        row["State_clean"],
        row["District_clean"],
        row["Crop_clean"]
    ),
    axis=1
)

top_candidates["demo25_records"] = top_candidates.apply(
    lambda row: get_apy_count(
        apy_2025_demo,
        row["State_clean"],
        row["District_clean"],
        row["Crop_clean"]
    ),
    axis=1
)

display(
    top_candidates[
        [
            "State",
            "District Name",
            "Commodity",
            "unique_days",
            "demo24_records",
            "demo25_records"
        ]
    ]
)

,State,District Name,Commodity,unique_days,demo24_records,demo25_records
0,West Bengal,Bankura,Wheat,365,1,1
1,Uttar Pradesh,Mainpuri,Wheat,365,1,1
2,West Bengal,Birbhum,Wheat,364,1,1
3,Uttar Pradesh,Fatehpur,Wheat,364,1,1
4,Uttar Pradesh,Gonda,Banana,362,1,1
5,Uttar Pradesh,Hardoi,Wheat,362,1,1
6,Uttar Pradesh,Shahjahanpur,Wheat,362,1,1
7,West Bengal,Paschim Bardhaman,Wheat,362,1,1
8,Uttar Pradesh,Balrampur,Wheat,360,1,1
9,Uttar Pradesh,Shamli,Garlic,358,1,1


In [11]:
# Inspect APY records for the top 4 candidates

candidates_to_check = [
    ("West Bengal", "Bankura", "Wheat"),
    ("Uttar Pradesh", "Mainpuri", "Wheat"),
    ("West Bengal", "Birbhum", "Wheat"),
    ("Uttar Pradesh", "Fatehpur", "Wheat")
]

for state, district, crop in candidates_to_check:
    print("\n" + "=" * 70)
    print(f"{state} | {district} | {crop}")
    print("=" * 70)

    print("\n2024 Demo:")
    display(
        apy_2024_demo[
            (apy_2024_demo["State"].str.strip().str.lower() == state.lower()) &
            (apy_2024_demo["District "].str.strip().str.lower() == district.lower()) &
            (apy_2024_demo["Crop"].str.strip().str.lower() == crop.lower())
        ]
    )

    print("\n2025 Demo:")
    display(
        apy_2025_demo[
            (apy_2025_demo["State"].str.strip().str.lower() == state.lower()) &
            (apy_2025_demo["District "].str.strip().str.lower() == district.lower()) &
            (apy_2025_demo["Crop"].str.strip().str.lower() == crop.lower())
        ]
    )


West Bengal | Bankura | Wheat

2024 Demo:


,State,District,Crop,Crop_Year,Season,Area,Production,Yield,Demo_Year
18284,West Bengal,BANKURA,Wheat,2018,Rabi,2254.0,5788.0,2.57,2024



2025 Demo:


,State,District,Crop,Crop_Year,Season,Area,Production,Yield,Demo_Year
19246,West Bengal,BANKURA,Wheat,2019,Rabi,1986.0,3678.0,1.85,2025



Uttar Pradesh | Mainpuri | Wheat

2024 Demo:


...

,State,District,Crop,Crop_Year,Season,Area,Production,Yield,Demo_Year
19247,West Bengal,BIRBHUM,Wheat,2019,Rabi,27607.0,82865.0,3.0,2025



Uttar Pradesh | Fatehpur | Wheat

2024 Demo:


,State,District,Crop,Crop_Year,Season,Area,Production,Yield,Demo_Year
17351,Uttar Pradesh,FATEHPUR,Wheat,2018,Rabi,172432.0,687681.0,3.99,2024



2025 Demo:


,State,District,Crop,Crop_Year,Season,Area,Production,Yield,Demo_Year
18324,Uttar Pradesh,FATEHPUR,Wheat,2019,Rabi,179472.0,710602.0,3.96,2025


In [12]:
temp = df[
    (df["State"].str.strip().str.lower() == "west bengal") &
    (df["District Name"].str.strip().str.lower() == "bankura") &
    (df["Commodity"].str.strip().str.lower() == "wheat")
]

print(temp["Market Name"].value_counts())

Market Name
Bishnupur(Bankura)    364
Khatra                314
Name: count, dtype: int64


In [13]:
series = df[
    (df["State"].str.strip().str.lower() == "west bengal") &
    (df["District Name"].str.strip().str.lower() == "bankura") &
    (df["Market Name"].str.strip().str.lower() == "bishnupur(bankura)") &
    (df["Commodity"].str.strip().str.lower() == "wheat")
].copy()

print("Rows:", len(series))
print("Unique dates:", series["Price Date"].nunique())
print("Date range:", series["Price Date"].min(), "to", series["Price Date"].max())

print("\nDuplicate dates:")
print(series["Price Date"].duplicated().sum())

print("\nMissing dates:")
all_dates = pd.date_range(
    series["Price Date"].min(),
    series["Price Date"].max(),
    freq="D"
)

missing_dates = all_dates.difference(series["Price Date"])

print("Number of missing calendar dates:", len(missing_dates))
print("First missing dates:", missing_dates[:20].tolist())

Rows: 364
Unique dates: 364
Date range: 01 Apr 2025 to 31 Oct 2024

Duplicate dates:
0

Missing dates:
Number of missing calendar dates: 0
First missing dates: []


In [15]:
master_df["Price Date"] = pd.to_datetime(
    master_df["Price Date"],
    format="%d %b %Y",
    errors="coerce"
)

master_df = master_df.sort_values("Price Date").reset_index(drop=True)

print("Date dtype:", master_df["Price Date"].dtype)
print("Start:", master_df["Price Date"].min())
print("End:", master_df["Price Date"].max())

display(master_df.head(10))
display(master_df.tail(10))

Date dtype: datetime64[us]
Start: 2024-08-15 00:00:00
End: 2025-08-14 00:00:00


,State,District Name,Market Name,Commodity,Price Date,Min Price (Rs./Quintal),Max Price (Rs./Quintal),Modal Price (Rs./Quintal)
0,West Bengal,Bankura,Bishnupur(Bankura),Wheat,2024-08-15,2700.0,3000.0,2850.0
1,West Bengal,Bankura,Bishnupur(Bankura),Wheat,2024-08-16,2700.0,3000.0,2800.0
2,West Bengal,Bankura,Bishnupur(Bankura),Wheat,2024-08-17,2700.0,3000.0,2800.0
3,West Bengal,Bankura,Bishnupur(Bankura),Wheat,2024-08-18,2750.0,3000.0,2800.0
4,West Bengal,Bankura,Bishnupur(Bankura),Wheat,2024-08-19,2750.0,3000.0,2800.0
5,West Bengal,Bankura,Bishnupur(Bankura),Wheat,2024-08-20,2750.0,3000.0,2850.0
6,West Bengal,Bankura,Bishnupur(Bankura),Wheat,2024-08-21,2750.0,3000.0,2850.0
7,West Bengal,Bankura,Bishnupur(Bankura),Wheat,2024-08-22,2700.0,3000.0,2800.0
8,West Bengal,Bankura,Bishnupur(Bankura),Wheat,2024-08-23,2700.0,3000.0,2800.0
9,West Bengal,Bankura,Bishnupur(Bankura),Wheat,2024-08-24,2700.0,3000.0,2800.0


,State,District Name,Market Name,Commodity,Price Date,Min Price (Rs./Quintal),Max Price (Rs./Quintal),Modal Price (Rs./Quintal)
354,West Bengal,Bankura,Bishnupur(Bankura),Wheat,2025-08-05,2800.0,3200.0,3000.0
355,West Bengal,Bankura,Bishnupur(Bankura),Wheat,2025-08-06,2800.0,3200.0,3000.0
356,West Bengal,Bankura,Bishnupur(Bankura),Wheat,2025-08-07,2800.0,3200.0,3000.0
357,West Bengal,Bankura,Bishnupur(Bankura),Wheat,2025-08-08,2800.0,3200.0,3000.0
358,West Bengal,Bankura,Bishnupur(Bankura),Wheat,2025-08-09,2800.0,3200.0,3000.0
359,West Bengal,Bankura,Bishnupur(Bankura),Wheat,2025-08-10,2800.0,3200.0,3000.0
360,West Bengal,Bankura,Bishnupur(Bankura),Wheat,2025-08-11,2800.0,3200.0,3000.0
361,West Bengal,Bankura,Bishnupur(Bankura),Wheat,2025-08-12,2800.0,3200.0,3000.0
362,West Bengal,Bankura,Bishnupur(Bankura),Wheat,2025-08-13,2800.0,3200.0,3000.0
363,West Bengal,Bankura,Bishnupur(Bankura),Wheat,2025-08-14,2800.0,3300.0,3000.0


In [14]:
master_df = series[
    [
        "State",
        "District Name",
        "Market Name",
        "Commodity",
        "Price Date",
        "Min Price (Rs./Quintal)",
        "Max Price (Rs./Quintal)",
        "Modal Price (Rs./Quintal)"
    ]
].copy()

master_df = master_df.sort_values("Price Date").reset_index(drop=True)

print("Shape:", master_df.shape)
display(master_df.head())
display(master_df.tail())

Shape: (364, 8)


,State,District Name,Market Name,Commodity,Price Date,Min Price (Rs./Quintal),Max Price (Rs./Quintal),Modal Price (Rs./Quintal)
0,West Bengal,Bankura,Bishnupur(Bankura),Wheat,01 Apr 2025,2800.0,3100.0,2900.0
1,West Bengal,Bankura,Bishnupur(Bankura),Wheat,01 Aug 2025,2800.0,3200.0,3000.0
2,West Bengal,Bankura,Bishnupur(Bankura),Wheat,01 Dec 2024,2600.0,2800.0,2700.0
3,West Bengal,Bankura,Bishnupur(Bankura),Wheat,01 Feb 2025,2700.0,3000.0,2800.0
4,West Bengal,Bankura,Bishnupur(Bankura),Wheat,01 Jan 2025,2750.0,3000.0,2850.0


,State,District Name,Market Name,Commodity,Price Date,Min Price (Rs./Quintal),Max Price (Rs./Quintal),Modal Price (Rs./Quintal)
359,West Bengal,Bankura,Bishnupur(Bankura),Wheat,31 Jan 2025,2700.0,3000.0,2800.0
360,West Bengal,Bankura,Bishnupur(Bankura),Wheat,31 Jul 2025,2800.0,3200.0,3000.0
361,West Bengal,Bankura,Bishnupur(Bankura),Wheat,31 Mar 2025,2800.0,3100.0,2900.0
362,West Bengal,Bankura,Bishnupur(Bankura),Wheat,31 May 2025,2800.0,3200.0,3000.0
363,West Bengal,Bankura,Bishnupur(Bankura),Wheat,31 Oct 2024,2700.0,3000.0,2800.0


In [19]:
master_df.isnull().sum()
df.isnull().sum()

Sl no.                       0
District Name                0
Market Name                  0
Commodity                    0
Variety                      0
Grade                        0
Min Price (Rs./Quintal)      0
Max Price (Rs./Quintal)      0
Modal Price (Rs./Quintal)    0
Price Date                   0
State                        0
dtype: int64

In [20]:
master_df["Year"] = master_df["Price Date"].dt.year
master_df["Month"] = master_df["Price Date"].dt.month
master_df["Month_Name"] = master_df["Price Date"].dt.month_name()
master_df["Quarter"] = master_df["Price Date"].dt.quarter
master_df["Day_of_Week"] = master_df["Price Date"].dt.dayofweek
master_df["Day_Name"] = master_df["Price Date"].dt.day_name()
master_df["Day_of_Year"] = master_df["Price Date"].dt.dayofyear

print(master_df[
    [
        "Price Date",
        "Year",
        "Month",
        "Month_Name",
        "Quarter",
        "Day_of_Week",
        "Day_Name",
        "Day_of_Year"
    ]
].head(10))

  Price Date  Year  Month Month_Name  Quarter  Day_of_Week   Day_Name  \
0 2024-08-15  2024      8     August        3            3   Thursday   
1 2024-08-16  2024      8     August        3            4     Friday   
2 2024-08-17  2024      8     August        3            5   Saturday   
3 2024-08-18  2024      8     August        3            6     Sunday   
4 2024-08-19  2024      8     August        3            0     Monday   
5 2024-08-20  2024      8     August        3            1    Tuesday   
6 2024-08-21  2024      8     August        3            2  Wednesday   
7 2024-08-22  2024      8     August        3            3   Thursday   
8 2024-08-23  2024      8     August        3            4     Friday   
9 2024-08-24  2024      8     August        3            5   Saturday   

   Day_of_Year  
0          228  
1          229  
2          230  
3          231  
4          232  
5          233  
6          234  
7          235  
8          236  
9          237  


In [21]:
def get_price_season(month):
    if month in [6, 7, 8, 9, 10]:
        return "Kharif"
    elif month in [11, 12, 1, 2]:
        return "Rabi"
    else:
        return "Summer"


master_df["Price_Season"] = master_df["Month"].apply(get_price_season)

# Wheat's agricultural season according to our APY data
master_df["Crop_Season"] = "Rabi"

print(
    master_df[
        ["Price Date", "Year", "Month", "Price_Season", "Crop_Season"]
    ].head(20)
)

print("\nPrice season distribution:")
print(master_df["Price_Season"].value_counts())

   Price Date  Year  Month Price_Season Crop_Season
0  2024-08-15  2024      8       Kharif        Rabi
1  2024-08-16  2024      8       Kharif        Rabi
2  2024-08-17  2024      8       Kharif        Rabi
3  2024-08-18  2024      8       Kharif        Rabi
4  2024-08-19  2024      8       Kharif        Rabi
5  2024-08-20  2024      8       Kharif        Rabi
6  2024-08-21  2024      8       Kharif        Rabi
7  2024-08-22  2024      8       Kharif        Rabi
8  2024-08-23  2024      8       Kharif        Rabi
9  2024-08-24  2024      8       Kharif        Rabi
10 2024-08-25  2024      8       Kharif        Rabi
11 2024-08-26  2024      8       Kharif        Rabi
12 2024-08-27  2024      8       Kharif        Rabi
13 2024-08-28  2024      8       Kharif        Rabi
14 2024-08-29  2024      8       Kharif        Rabi
15 2024-08-30  2024      8       Kharif        Rabi
16 2024-08-31  2024      8       Kharif        Rabi
17 2024-09-01  2024      9       Kharif        Rabi
18 2024-09-0

In [22]:
# Get the APY information for our selected State + District + Crop

apy_24 = apy_2024_demo[
    (apy_2024_demo["State"].str.strip().str.lower() == "west bengal") &
    (apy_2024_demo["District "].str.strip().str.lower() == "bankura") &
    (apy_2024_demo["Crop"].str.strip().str.lower() == "wheat")
].copy()

apy_25 = apy_2025_demo[
    (apy_2025_demo["State"].str.strip().str.lower() == "west bengal") &
    (apy_2025_demo["District "].str.strip().str.lower() == "bankura") &
    (apy_2025_demo["Crop"].str.strip().str.lower() == "wheat")
].copy()

print("2024 Demo APY:")
display(apy_24)

print("\n2025 Demo APY:")
display(apy_25)

2024 Demo APY:


,State,District,Crop,Crop_Year,Season,Area,Production,Yield,Demo_Year
18284,West Bengal,BANKURA,Wheat,2018,Rabi,2254.0,5788.0,2.57,2024



2025 Demo APY:


,State,District,Crop,Crop_Year,Season,Area,Production,Yield,Demo_Year
19246,West Bengal,BANKURA,Wheat,2019,Rabi,1986.0,3678.0,1.85,2025


In [24]:
print(apy_24.columns.tolist())
print(apy_25.columns.tolist())

['State', 'District ', 'Crop', 'Crop_Year', 'Season', 'Area ', 'Production', 'Yield', 'Demo_Year']
['State', 'District ', 'Crop', 'Crop_Year', 'Season', 'Area ', 'Production', 'Yield', 'Demo_Year']


In [25]:
print("master_df columns:")
print(master_df.columns.tolist())

master_df columns:
['State', 'District Name', 'Market Name', 'Commodity', 'Price Date', 'Min Price (Rs./Quintal)', 'Max Price (Rs./Quintal)', 'Modal Price (Rs./Quintal)', 'Year', 'Month', 'Month_Name', 'Quarter', 'Day_of_Week', 'Day_Name', 'Day_of_Year', 'Price_Season', 'Crop_Season']


In [26]:
apy_24.columns = apy_24.columns.str.strip()
apy_25.columns = apy_25.columns.str.strip()

print(apy_24.columns.tolist())
print(apy_25.columns.tolist())

['State', 'District', 'Crop', 'Crop_Year', 'Season', 'Area', 'Production', 'Yield', 'Demo_Year']
['State', 'District', 'Crop', 'Crop_Year', 'Season', 'Area', 'Production', 'Yield', 'Demo_Year']


In [27]:
master_df["Area"] = master_df["Year"].map({
    2024: apy_24.iloc[0]["Area"],
    2025: apy_25.iloc[0]["Area"]
})

master_df["Production"] = master_df["Year"].map({
    2024: apy_24.iloc[0]["Production"],
    2025: apy_25.iloc[0]["Production"]
})

master_df["Yield"] = master_df["Year"].map({
    2024: apy_24.iloc[0]["Yield"],
    2025: apy_25.iloc[0]["Yield"]
})

print(master_df[["Year", "Area", "Production", "Yield"]].head())

   Year    Area  Production  Yield
0  2024  2254.0      5788.0   2.57
1  2024  2254.0      5788.0   2.57
2  2024  2254.0      5788.0   2.57
3  2024  2254.0      5788.0   2.57
4  2024  2254.0      5788.0   2.57


In [30]:
import requests

print("requests imported successfully")

latitude = 23.2324
longitude = 87.0784

print("Latitude:", latitude)
print("Longitude:", longitude)

requests imported successfully
Latitude: 23.2324
Longitude: 87.0784


In [31]:
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": latitude,
    "longitude": longitude,
    "start_date": "2024-08-15",
    "end_date": "2025-08-14",
    "daily": "precipitation_sum,temperature_2m_max,temperature_2m_min",
    "timezone": "Asia/Kolkata"
}

response = requests.get(url, params=params)

print("Status code:", response.status_code)

Status code: 200


In [32]:
weather_data = response.json()

weather_df = pd.DataFrame({
    "Price Date": pd.to_datetime(weather_data["daily"]["time"]),
    "Rainfall_mm": weather_data["daily"]["precipitation_sum"],
    "Max_Temp_C": weather_data["daily"]["temperature_2m_max"],
    "Min_Temp_C": weather_data["daily"]["temperature_2m_min"]
})

print(weather_df.shape)
print(weather_df.head())

(365, 4)
  Price Date  Rainfall_mm  Max_Temp_C  Min_Temp_C
0 2024-08-15          8.3        32.8        25.3
1 2024-08-16          8.9        32.8        25.1
2 2024-08-17         11.5        32.2        24.7
3 2024-08-18          8.2        31.7        25.2
4 2024-08-19         12.2        30.8        25.2


In [33]:
master_df = master_df.merge(
    weather_df,
    on="Price Date",
    how="left"
)

print(master_df.shape)
print(master_df[[
    "Price Date",
    "Modal Price (Rs./Quintal)",
    "Rainfall_mm",
    "Max_Temp_C",
    "Min_Temp_C"
]].head())

(364, 23)
  Price Date  Modal Price (Rs./Quintal)  Rainfall_mm  Max_Temp_C  Min_Temp_C
0 2024-08-15                     2850.0          8.3        32.8        25.3
1 2024-08-16                     2800.0          8.9        32.8        25.1
2 2024-08-17                     2800.0         11.5        32.2        24.7
3 2024-08-18                     2800.0          8.2        31.7        25.2
4 2024-08-19                     2800.0         12.2        30.8        25.2


In [34]:
print(master_df[[
    "Rainfall_mm",
    "Max_Temp_C",
    "Min_Temp_C"
]].isna().sum())


Rainfall_mm    0
Max_Temp_C     0
Min_Temp_C     0
dtype: int64


In [35]:
master_df["Lag_1"] = master_df["Modal Price (Rs./Quintal)"].shift(1)
master_df["Lag_7"] = master_df["Modal Price (Rs./Quintal)"].shift(7)
master_df["Lag_14"] = master_df["Modal Price (Rs./Quintal)"].shift(14)
master_df["Lag_30"] = master_df["Modal Price (Rs./Quintal)"].shift(30)

print(master_df[[
    "Price Date",
    "Modal Price (Rs./Quintal)",
    "Lag_1",
    "Lag_7",
    "Lag_14",
    "Lag_30"
]].head(35))

   Price Date  Modal Price (Rs./Quintal)   Lag_1   Lag_7  Lag_14  Lag_30
0  2024-08-15                     2850.0     NaN     NaN     NaN     NaN
1  2024-08-16                     2800.0  2850.0     NaN     NaN     NaN
2  2024-08-17                     2800.0  2800.0     NaN     NaN     NaN
3  2024-08-18                     2800.0  2800.0     NaN     NaN     NaN
4  2024-08-19                     2800.0  2800.0     NaN     NaN     NaN
5  2024-08-20                     2850.0  2800.0     NaN     NaN     NaN
6  2024-08-21                     2850.0  2850.0     NaN     NaN     NaN
7  2024-08-22                     2800.0  2850.0  2850.0     NaN     NaN
8  2024-08-23                     2800.0  2800.0  2800.0     NaN     NaN
9  2024-08-24                     2800.0  2800.0  2800.0     NaN     NaN
10 2024-08-25                     2800.0  2800.0  2800.0     NaN     NaN
11 2024-08-26                     2800.0  2800.0  2800.0     NaN     NaN
12 2024-08-27                     2800.0  2800.0  2

In [36]:
price_col = "Modal Price (Rs./Quintal)"

master_df["Rolling_Mean_7"] = (
    master_df[price_col].shift(1).rolling(7).mean()
)

master_df["Rolling_Mean_14"] = (
    master_df[price_col].shift(1).rolling(14).mean()
)

master_df["Rolling_Mean_30"] = (
    master_df[price_col].shift(1).rolling(30).mean()
)

print(master_df[[
    "Price Date",
    price_col,
    "Rolling_Mean_7",
    "Rolling_Mean_14",
    "Rolling_Mean_30"
]].tail())

    Price Date  Modal Price (Rs./Quintal)  Rolling_Mean_7  Rolling_Mean_14  \
359 2025-08-10                     3000.0          3000.0           3000.0   
360 2025-08-11                     3000.0          3000.0           3000.0   
361 2025-08-12                     3000.0          3000.0           3000.0   
362 2025-08-13                     3000.0          3000.0           3000.0   
363 2025-08-14                     3000.0          3000.0           3000.0   

     Rolling_Mean_30  
359           3000.0  
360           3000.0  
361           3000.0  
362           3000.0  
363           3000.0  


In [37]:
print(master_df[[
    "Lag_1",
    "Lag_7",
    "Lag_14",
    "Lag_30",
    "Rolling_Mean_7",
    "Rolling_Mean_14",
    "Rolling_Mean_30"
]].isna().sum())

Lag_1               1
Lag_7               7
Lag_14             14
Lag_30             30
Rolling_Mean_7      7
Rolling_Mean_14    14
Rolling_Mean_30    30
dtype: int64


In [38]:
feature_cols = [
    "Lag_1",
    "Lag_7",
    "Lag_14",
    "Lag_30",
    "Rolling_Mean_7",
    "Rolling_Mean_14",
    "Rolling_Mean_30"
]

master_df = master_df.dropna(subset=feature_cols).reset_index(drop=True)

print("New shape:", master_df.shape)
print("Remaining missing values:")
print(master_df[feature_cols].isna().sum().sum())


New shape: (334, 30)
Remaining missing values:
0


In [39]:
feature_cols = [
    "Lag_1",
    "Lag_7",
    "Lag_14",
    "Lag_30",
    "Rolling_Mean_7",
    "Rolling_Mean_14",
    "Rolling_Mean_30",
    "Rainfall_mm",
    "Max_Temp_C",
    "Min_Temp_C",
    "Area",
    "Production",
    "Yield",
    "Month",
    "Quarter",
    "Day_of_Week",
    "Day_of_Year"
]

X = master_df[feature_cols]
y = master_df["Modal Price (Rs./Quintal)"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Features:", X.columns.tolist())

X shape: (334, 17)
y shape: (334,)
Features: ['Lag_1', 'Lag_7', 'Lag_14', 'Lag_30', 'Rolling_Mean_7', 'Rolling_Mean_14', 'Rolling_Mean_30', 'Rainfall_mm', 'Max_Temp_C', 'Min_Temp_C', 'Area', 'Production', 'Yield', 'Month', 'Quarter', 'Day_of_Week', 'Day_of_Year']


In [40]:
split_index = int(len(master_df) * 0.8)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

print("Training period:")
print(master_df["Price Date"].iloc[0], "to", master_df["Price Date"].iloc[split_index - 1])

print("\nTesting period:")
print(master_df["Price Date"].iloc[split_index], "to", master_df["Price Date"].iloc[-1])

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

Training period:
2024-09-14 00:00:00 to 2025-06-08 00:00:00

Testing period:
2025-06-09 00:00:00 to 2025-08-14 00:00:00

Training samples: 267
Testing samples: 67
